MD# Caminho 2: SQL no BigQuery ML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/souuzaa/gdg-blumenau-ml-gcp/blob/main/02-bigquery-ml/bqml_penguins.ipynb)

O mesmo modelo dos pinguins, treinado dentro do BigQuery. Funciona no **BigQuery sandbox** (sem cartão).

Antes: crie um projeto em [console.cloud.google.com](https://console.cloud.google.com) e abra o BigQuery uma vez para aceitar os termos.

MD## 1. Autenticar e escolher o projeto

In [ ]:
PROJECT_ID = "seu-project-id"  # @param {type:"string"}
DATASET = "gdg_blumenau"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    print("Fora do Colab: use 'gcloud auth application-default login' no terminal.")

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# A tabela pública dos pinguins fica em US. O dataset precisa estar na mesma localização.
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = "US"
client.create_dataset(ds, exists_ok=True)

def rodar(sql):
    """Roda o SQL e devolve um DataFrame (vazio para CREATE/DROP)."""
    job = client.query(sql)
    resultado = job.result()
    print(f"{job.statement_type}: {(job.total_bytes_processed or 0) / 1e6:.2f} MB processados")
    return resultado.to_dataframe() if resultado.total_rows else None

T = f"`{PROJECT_ID}.{DATASET}"  # prefixo das tabelas

MD## 2. Base com split fixo

80% treino, 20% teste. O hash deixa a divisão estável entre execuções.

In [ ]:
rodar(f"""
CREATE OR REPLACE VIEW {T}.pinguins_base` AS
SELECT species, island, sex, culmen_length_mm, culmen_depth_mm, flipper_length_mm, body_mass_g,
  IF(MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(p))), 10) < 8, 'treino', 'teste') AS split
FROM `bigquery-public-data.ml_datasets.penguins` AS p
WHERE culmen_length_mm IS NOT NULL AND culmen_depth_mm IS NOT NULL
  AND flipper_length_mm IS NOT NULL AND body_mass_g IS NOT NULL
""")
rodar(f"SELECT split, species, COUNT(*) AS n FROM {T}.pinguins_base` GROUP BY 1, 2 ORDER BY 1, 2")

MD## 3. Treinar com `CREATE MODEL`

Leva em torno de um minuto.

In [ ]:
rodar(f"""
CREATE OR REPLACE MODEL {T}.pinguins_especie`
OPTIONS (model_type = 'LOGISTIC_REG', input_label_cols = ['species'], data_split_method = 'NO_SPLIT') AS
SELECT species, culmen_length_mm, culmen_depth_mm, flipper_length_mm, body_mass_g
FROM {T}.pinguins_base`
WHERE split = 'treino'
""")

MD## 4. Avaliar em pinguins que o modelo nunca viu

In [ ]:
TESTE = f"""(SELECT species, culmen_length_mm, culmen_depth_mm, flipper_length_mm, body_mass_g
             FROM {T}.pinguins_base` WHERE split = 'teste')"""

display(rodar(f"SELECT * FROM ML.EVALUATE(MODEL {T}.pinguins_especie`, {TESTE})"))
display(rodar(f"SELECT * FROM ML.CONFUSION_MATRIX(MODEL {T}.pinguins_especie`, {TESTE})"))

MD## 5. Prever

As previsões menos confiantes primeiro: são os casos mais interessantes.

In [ ]:
rodar(f"""
SELECT species AS real, predicted_species AS previsto,
  ROUND((SELECT prob FROM UNNEST(predicted_species_probs) WHERE label = predicted_species), 3) AS confianca
FROM ML.PREDICT(MODEL {T}.pinguins_especie`, {TESTE})
ORDER BY confianca
LIMIT 10
""")

In [ ]:
# Um pinguim novo. Mude os números.
rodar(f"""
SELECT predicted_species, predicted_species_probs
FROM ML.PREDICT(MODEL {T}.pinguins_especie`,
  (SELECT 45.0 AS culmen_length_mm, 15.5 AS culmen_depth_mm, 215.0 AS flipper_length_mm, 5000.0 AS body_mass_g))
""")

MD## 6. Limpar

No sandbox tudo expira em 60 dias, mas é bom hábito.

In [ ]:
rodar(f"DROP MODEL IF EXISTS {T}.pinguins_especie`; DROP VIEW IF EXISTS {T}.pinguins_base`;")

MD## Experimentos

1. Troque `LOGISTIC_REG` por `BOOSTED_TREE_CLASSIFIER` e compare o `ML.EVALUATE`.
2. Inclua `island` e `sex` no `SELECT` do treino. O BigQuery ML faz o one-hot encoding sozinho.